## Quantum Machine Learning

Es como ML clásico pero sustituye una de las partes del pipeline.

En este notebook aprenderemos a:
1. **Comprar** machine learning clásico y QML
2. **Añadir capas** y mejorar un modelo de QML

In [ ]:
!pip install qiskit qiskit-aer qiskit-algorithms

In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import numpy as np

dataset = load_iris()
X_full = dataset.data
y_full = dataset.target

# inary: setosa (0) vs versicolor (1) -> usamos petal length y petal width (las más discriminativas)
mask = y_full < 2
X_raw = X_full[mask, 2:4]  #features 2 y 3: petal length, petal width
y_all = y_full[mask]

#Normalizar a [0, 1] para la codificación cuántica
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_raw)

print("Dataset: Iris (setosa vs versicolor)")
print("Features: longitud del pétalo, ancho del pétalo")
print(f"Muestras: {len(X_scaled)} | Clase 0 (setosa): {(y_all==0).sum()} | Clase 1 (versicolor): {(y_all==1).sum()}")
print("\nPrimeras 5 muestras (normalizadas):")
print("  petal_len  petal_wid  label")
for i in range(5):
    print(f"  {X_scaled[i,0]:.3f}      {X_scaled[i,1]:.3f}      {y_all[i]}")

#Train/test split estratificado
indices = list(range(len(X_scaled)))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=y_all)

X = [(float(X_scaled[i, 0]), float(X_scaled[i, 1])) for i in range(len(X_scaled))]
y = y_all.tolist()

print(f"\nTrain: {len(train_idx)} muestras | Test: {len(test_idx)} muestras")

Mientras más parámetros de entrada, más qubits necesita el [circuito](https://quantum.cloud.ibm.com/composer?initial=N4IgjghgzgtiBcIDyAFAogOQIoEEDKAsgAQBMAdAAwDcAOgHYCWdAxgDYCuAJgKZE3jdWDAEYBGMk2b9ademABO3AOZEwAbQAsAXRnNFK5pp315ATwAUoogCoiABwYBKVWorG68gF6Wb9py7cZMx9bB2d1QPoYbmh2RQCtIgBaAD4iQ0i6EAAaEDoIaIQQAFU6ABcGMtZuTnSGeWZ2SpAAXyA) o más operaciones sobre el qubit en el que se trabaje.
Dependiendo del número de clases, se puede plantear el clasificador así:

| Caso | Estrategia | Cómo funciona | Complejidad |
|---|---|---|---|
| **2 clases** | **Un solo circuito binario** | El circuito devuelve `0` o `1` (o probabilidad de clase 1 y luego umbral). | Baja |
| **Más de 2 clases** | **OvR (One-vs-Rest)** | Entrenas varios clasificadores binarios: uno por clase (`clase_i` vs resto). | Media (sencillo de implementar) |
| **Más de 2 clases** | **Un único circuito multiclase** | Usas un circuito con más salidas (por ejemplo, varios qubits) y aplicas **softmax** para mapear a probabilidades por clase. | Alta (requiere rediseñar el circuito) |

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import numpy as np

simulator = AerSimulator()

def quantum_classifier(x1, x2, theta, shots=512): #El mejor angulo theta es el que maximiza la probabilidad de clasificar correctamente los datos
    qc = QuantumCircuit(1, 1)
    qc.ry(x1 * np.pi, 0)   #Se codifica en el ángulo de rotación RY los datos de entrada X1
    qc.rz(x2 * np.pi, 0)   #Se codifica en el ángulo de rotación RZ los datos de entrada X2 -> segunda capa
    qc.ry(theta, 0)        #Se aplica una rotación RY con un ángulo de parámetro theta que se ajustará durante el entrenamiento
    qc.measure(0, 0)

    result = simulator.run(qc, shots=shots).result()
    counts = result.get_counts(qc)
    prob_1 = counts.get("1", 0) / shots
    return prob_1

---

## Que hace cada parte del circuito

![Circuito QML](https://raw.githubusercontent.com/jorgecs/apuntes/main/docs/ut5_ia_aplicada/1_transformers/img/circuito_qml.png)

### 1) `RY(x)` / `RZ(x)` (encoding de datos)
`RY(x * pi)` o `RZ(x * pi)` convierte cada feature de entrada en una rotacion del qubit.
- Si `x` cambia, cambia el estado cuantico.
- Es el equivalente a "meter los datos" en el modelo.
- Es comun combinar capas `RY` y `RZ` para aumentar expresividad.


### 2) `RY(theta)` (parametro entrenable)
`theta` NO viene del dataset: se aprende durante el entrenamiento.
- Igual que un peso en ML clasico.
- El optimizador (ADAM) ajusta `theta` para minimizar la perdida.

### 3) Entrelazamiento (`CX`, `CZ`, ...)
Cuando hay varios qubits, el entrelazamiento permite que los qubits no se comporten de forma independiente.
- Captura interacciones entre features.
- Suele ayudar cuando la frontera de decision es mas compleja o no lineal.
- Hace que un qubit dependa de otro. NO es exactamente un `if` clasico; es una correlacion cuantica entre estados.


### 4) Capas del circuito
Pensad el circuito como una red neuronal por bloques:
1. Capa de encoding: puertas que dependen de `x` (`RY`, `RZ`).
2. Capa variacional: puertas con parametros entrenables (`theta`).
3. Capa de entrelazamiento: conecta qubits (`CX`, `CZ`).
4. Repeticion de bloques: mas expresividad (como mas profundidad en una red).

### 5) Medicion y prediccion
Al medir, obtenemos conteos de `0/1` y estimamos `p(1)`.
Luego aplicamos umbral (por ejemplo, 0.5) para decidir la clase.
- `shots`: numero de mediciones para estimar probabilidades. Es típico tener cientos o miles de `shots` para obtener la tendencia de los valores obtenidos (la computación cuántica es probabilística, no determinista).

> **IMPORTANTE**: `RY(x)`/`RZ(x)` codifican informacion, `RY(theta)` aprende, el entrelazamiento mezcla informacion (si hay 2+ qubits), y las capas repetidas aumentan la capacidad del modelo.

### 6) Optimización del circuito: usamos **ADAM** como en ML clásico, minimizando la pérdida en train, pero en este caso, sobre los parámetros del circuito (`theta`).

In [15]:
#Entrenamiento del modelo cuántico
from qiskit_algorithms.optimizers import ADAM

def objective(x):
    theta = float(x[0])
    losses = []
    for i in train_idx:
        x1, x2 = X[i]
        p1 = quantum_classifier(x1, x2, theta, shots=512)
        losses.append((p1 - y[i]) ** 2)
    return float(np.mean(losses))

np.random.seed(42)
theta_init = np.array([np.random.uniform(0, 2 * np.pi)])
optimizer = ADAM(maxiter=60, lr=0.10)

result = optimizer.minimize(fun=objective, x0=theta_init)

best_theta = float(result.x[0]) % (2 * np.pi)
best_loss = float(result.fun)

def effective_p1(p1, invert_labels=False):
    return 1.0 - p1 if invert_labels else p1

#Detecta si el mapeo aprendido está invertido (clase 0<->1 intercambiada). En ML clásico esto lo hace automáticamente la librería
correct_normal = 0
correct_inverted = 0
for i in train_idx:
    x1, x2 = X[i]
    p1 = quantum_classifier(x1, x2, best_theta, shots=512)

    pred_normal = 1 if p1 > 0.5 else 0
    pred_inverted = 1 if (1.0 - p1) > 0.5 else 0

    correct_normal += int(pred_normal == y[i])
    correct_inverted += int(pred_inverted == y[i])

invert_labels = correct_inverted > correct_normal

best_error = 0
conf_matrix = np.zeros((2, 2), dtype=int)
for i in train_idx:
    x1, x2 = X[i]
    p1 = quantum_classifier(x1, x2, best_theta, shots=512)
    p1_eff = effective_p1(p1, invert_labels=invert_labels)
    pred_label = 1 if p1_eff > 0.5 else 0

    best_error += abs(pred_label - y[i])
    conf_matrix[y[i], pred_label] += 1

print(f"Best theta: {best_theta:.4f} rad")
print(f"Train loss: {best_loss:.4f}")
print(f"invert_labels: {invert_labels} (normal={correct_normal}, inverted={correct_inverted})")
print(f"Train errors: {best_error}/{len(train_idx)}")
print("Confusion matrix (train):")
print(conf_matrix)

In [13]:
#Comparación: Cuántico vs Clásico
from sklearn.linear_model import LogisticRegression

X_np = np.array(X)
y_np = np.array(y)

clf = LogisticRegression()
clf.fit(X_np[train_idx], y_np[train_idx])

#Precisión cuántica con orientación calibrada (usa invert_labels de la celda de entrenamiento).
q_correct = sum(
    (
        1
        if effective_p1(
            quantum_classifier(X[i][0], X[i][1], best_theta, shots=2048),
            invert_labels=invert_labels
        ) > 0.5
        else 0
    ) == y[i]
    for i in test_idx
)
q_acc = q_correct / len(test_idx)

#Precisión del modelo clásico
c_acc = clf.score(X_np[test_idx], y_np[test_idx])

print(f"Quantum test accuracy (calibrated): {q_acc:.2%}")
print(f"Classical test accuracy:            {c_acc:.2%}")

---

## Ejercicio 1 - Mejorar los resultados de QML
Para mejorar los resultados del clasificador cuántico, existen varias formas:
- Aumentar la cantidad de ejecuciones (shots)
- Añadir más rotaciones (ry, rz)
- Añadir entrelazamiento entre qubits (cx, cz)

Cread una funcion quantum_classifier nueva cambiando el circuito interno y volved a ejecutar la celda de la función objetivo.

In [12]:
def quantum_classifier(x1, x2, theta, shots=None): #El mejor angulo theta es el que maximiza la probabilidad de clasificar correctamente los datos
    qc = QuantumCircuit(1, 1)
    qc.ry(x1 * np.pi, 0)   #Se codifica en el ángulo de rotación RY los datos de entrada X1
    qc.rz(x2 * np.pi, 0)   #Se codifica en el ángulo de rotación RZ los datos de entrada X2
    qc.ry(theta, 0)        #Se aplica una rotación RY con un ángulo de parámetro theta que se ajustará durante el entrenamiento
    qc.measure(0, 0)

    result = simulator.run(qc, shots=shots).result()
    counts = result.get_counts(qc)
    prob_1 = counts.get("1", 0) / shots
    return prob_1

In [14]:
def quantum_classifier(x1, x2, theta, shots=2048):
    qc = QuantumCircuit(2, 1)

    #Encoding de datos (ry/ry)
    qc.ry(np.pi * x1, 0)
    qc.ry(np.pi * x2, 1)

    #Capa de entrenamiento
    qc.ry(theta, 0)
    qc.ry(theta, 1)

    # ========== TODO: Añadir capa de entrelazamiento ==========
    

    # ========== TODO: Añadir nueva capa de encoding y entrenamiento(rz/ry) ==========

    #Medición del primer qubit para clasificación binaria
    qc.measure(1, 0)

    counts = simulator.run(qc, shots=shots).result().get_counts(qc)
    p1 = counts.get("1", 0) / shots
    return p1

---

## Ejercicio 2 - Entender algunas puertas cuánticas

Cambia las puertas a H y X para ver qué hacen.

In [ ]:
circuit = QuantumCircuit(1)
# ========== TODO: Añadir puerta cuántica ==========
circuit.measure_all()

circuit.draw("mpl")

result = simulator.run(circuit, shots=1024).result()
counts = result.get_counts(circuit)
print(counts)